## Model

### Load the data

In [ ]:
import pandas as pd

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path = "/content/drive/My Drive/wikipedia/wiki_extracts_with_labels.csv"

wiki = pd.read_csv(file_path)

wiki.head()

### Set Up the Zero-Shot Classification Pipeline

The model facebook/bart-large-mnli is a version of BART (Bidirectional and Auto-Regressive Transformers) fine-tuned on the Multi-Genre Natural Language Inference (MNLI) dataset.    
The model is trained to understand relationships between premises and hypotheses, which enables the following trick for classification:
You provide your input text as the "premise".
Each candidate label becomes a hypothesis, like:
"This text is about gender bias."
The model evaluates how entailment, neutral, or contradiction the relationship is between the premise and each hypothesis.
This allows it to predict whether a text implies a certain label — without any task-specific fine-tuning.

#### Experiment with the following type of biases

 Framing bias: Differences in how events/facts are presented or emphasized (e.g. choice of verbs/adjectives).  
Omission bias: Facts or viewpoints present in one language article but missing entirely in another.  
Coverage bias: Disparities in article length, section counts, or citation counts across languages.  
Source bias: Reliance on different types or credibility levels of sources (e.g. local vs. international).  
Ideological bias: Slant toward a particular political or philosophical viewpoint beyond mere “political bias.”  
Nationalistic bias: Excessive patriotism or negative framing of other countries.
Religious bias: Favorable/unfavorable language toward a religion or its adherents.  
Economic bias: Tendency to portray economic systems or companies in a consistently positive/negative light.  
Structural bias: Where key sections (e.g. History, Culture, Criticism) appear in one language but not another.  
Sensationalist bias: Overuse of dramatic or emotionally charged wording.

Removing some bias types:  
ideological bias	Broad term — overlaps heavily with political, religious, and economic bias.  
nationalist bias	Often a subtype of political bias or cultural bias.  religious bias	May be retained depending on your dataset focus — otherwise often part of cultural or ideological bias.  
economic bias	Frequently overlaps with ideological bias and may be rare in general Wikipedia text.  
structural bias	Difficult to detect in article-level text; better suited for analyzing entire editorial systems.  
sensationalist bias	More relevant in tabloid/news content than Wikipedia. Hard to detect in encyclopedic tone.

In [ ]:
from transformers import pipeline
from datasets import Dataset
import pandas as pd

# Initialize the zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

In [ ]:
# Define candidate labels for bias detection (these can be adjusted based on the bias types to be detected)
# candidate_labels = ["gender bias", "racial bias", "political bias", "age bias", "cultural bias", "framing bias", "omission bias", "coverage bias","source bias", "ideological bias","nationalist bias","religious bias","economic bias","structural bias","sensationalist bias"]
candidate_labels = ["gender bias", "racial bias", "political bias", "age bias", "cultural bias", "framing bias", "omission bias", "coverage bias"]

In [ ]:
# Loop through the articles and classify them
from tqdm import tqdm

wiki["bias_detection_result"] = None  # Create the column with default empty values

for index, row in tqdm(wiki.iterrows(), total=len(wiki)):
    article_text = row["extract"]
    # Skip empty or NaN text, code has broken down before, ther are empty ones
    if pd.isna(article_text) or not isinstance(article_text, str) or article_text.strip() == "":
        continue
    result = classifier(article_text, candidate_labels)
    wiki.at[index, "bias_detection_result"] = result

In [ ]:
wiki.head(5)

In [ ]:
wiki = wiki.drop(columns="average_bias_score") # drop this column at this step

In [ ]:
wiki.to_csv("/content/drive/My Drive/wikipedia/wiki_0512.csv", index=False) # save the file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import ast  # For safely parsing stringified dicts

# Load the CSV
wiki = pd.read_csv("/content/drive/My Drive/wikipedia/wiki_0512.csv")

# Convert string to dict
wiki["bias_detection_result"] = wiki["bias_detection_result"].apply(ast.literal_eval)

# Extract {label: score} as a new dict for each row
bias_expanded = wiki["bias_detection_result"].apply(
    lambda x: dict(zip(x["labels"], x["scores"]))
)

# Convert to DataFrame
bias_df = pd.DataFrame(bias_expanded.tolist())

# Compute the most dominant bias score (max score per row)
bias_df["max_bias"] = bias_df.max(axis=1)

# Compute the label with the highest bias score
bias_df["top_bias"] = bias_df.idxmax(axis=1)

# Combine with original DataFrame
wiki = pd.concat([wiki, bias_df], axis=1)

In [ ]:
wiki.sample(5)

In [ ]:
# Sort the DataFrame by 'max_bias' in descending order
bias_df_sorted = bias_df.sort_values(by="max_bias", ascending=False)

# Display the sorted DataFrame
bias_df_sorted.head()

#### Grouping biases

Group	Biases:  
Social Bias, including gender bias, racial bias, age bias  
Media Bias:	framing bias, omission bias, coverage bias

Why grouping biases:  
Improves classification confidence: Fewer, more distinct labels reduce overlap and confusion for the model.  
Simplifies output: Easier to interpret and act on three broad categories than eight specific ones.  

In [ ]:
from transformers import pipeline
from datasets import Dataset
import pandas as pd
from tqdm import tqdm

In [ ]:
candidate_labels = ["social bias", "political bias", "cultural bias", "media bias"]

In [ ]:
# Initialize the zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path = "/content/drive/My Drive/wikipedia/wiki_extracts_with_labels.csv"

wiki = pd.read_csv(file_path)

In [ ]:
wiki["bias_detection_result"] = None  # Create the column with default empty values

for index, row in tqdm(wiki.iterrows(), total=len(wiki)):
    article_text = row["extract"]
    # Skip empty or NaN text, code has broken down before, ther are empty ones
    if pd.isna(article_text) or not isinstance(article_text, str) or article_text.strip() == "":
        continue
    result = classifier(article_text, candidate_labels)
    wiki.at[index, "bias_detection_result"] = result

In [ ]:
# Loop through the articles and classify them

# Filter out invalid or empty text rows first
wiki = wiki[wiki["extract"].apply(lambda x: isinstance(x, str) and x.strip() != "")].copy()

wiki["bias_detection_result"] = None  # Create the column with default empty values

for index, row in tqdm(wiki.iterrows(), total=len(wiki)):
    article_text = row["extract"]
    # Skip empty or NaN text, code has broken down before, ther are empty ones
    if pd.isna(article_text) or not isinstance(article_text, str) or article_text.strip() == "":
        continue
    result = classifier(article_text, candidate_labels)
    wiki.at[index, "bias_detection_result"] = result

In [ ]:
wiki.head(5)

In [ ]:
wiki.to_csv("/content/drive/My Drive/wikipedia/wiki_4biases.csv", index=False) # save the file

In [ ]:
import ast  # For safely parsing stringified dicts

# Load the CSV
wiki = pd.read_csv("/content/drive/My Drive/wikipedia/wiki_4biases.csv")

# Convert string to dict
wiki["bias_detection_result"] = wiki["bias_detection_result"].apply(ast.literal_eval)

# Extract {label: score} as a new dict for each row
bias_expanded = wiki["bias_detection_result"].apply(
    lambda x: dict(zip(x["labels"], x["scores"]))
)

# Convert to DataFrame
bias_df = pd.DataFrame(bias_expanded.tolist())

# Compute the most dominant bias score (max score per row)
bias_df["max_bias"] = bias_df.max(axis=1)

# Compute the label with the highest bias score
bias_df["top_bias"] = bias_df.idxmax(axis=1)

# Combine with original DataFrame
wiki = pd.concat([wiki, bias_df], axis=1)

In [ ]:
wiki.head()

In [ ]:
# Sort the DataFrame by 'max_bias' in descending order
bias_df_sorted = bias_df.sort_values(by="max_bias", ascending=False)

# Display the sorted DataFrame
bias_df_sorted.head()

# xlsx csv export result

In [ ]:
# install the Excel engine
# !pip install openpyxl --quiet

output_path = "/content/drive/My Drive/wikipedia/wiki_bias_detection_results.xlsx"
# Export DataFrame to Excel
wiki.to_excel(output_path, index=False, engine="openpyxl")
print(f"✅ Bias detection results exported to: {output_path}")

# CSV export path
csv_output_path = "/content/drive/My Drive/wikipedia/wiki_bias_detection_results.csv"
# Export DataFrame to CSV
wiki.to_csv(csv_output_path, index=False)

print(f"✅ Bias detection results also exported to: {csv_output_path}")